# RAG query pipeline — weighted stratified retrieval

**Problem (corpus skew).** The unified index is dominated by English↔Japanese parallel chunks (`eng_jap_*`, tens of thousands of vectors). A single global `query_vectors` top‑k almost always returns only those rows, so **grammar** and **style** rulebooks (dozens to hundreds of chunks) rarely surface.

**Approach.** Encode the query **once**, then run **separate** `query_vectors` calls with a [metadata `filter`](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors-metadata-filtering.html) on `source_file`, so each stratum gets its own quota:

| Stratum | Top‑k | Role |
|---|---|---|
| `eng_jap` | 3 | Parallel translation examples |
| `gemini_annotated` | 1–2 (`GEMINI_TOP_K`) | Curated annotations |
| `grammar` | 1 | Structural rule |
| `style_guide` | 1 | Tone / formatting rule |

Results are merged in that order, **deduped by vector key**, then **chunk text is loaded only from the vector index**: first from metadata returned with `query_vectors`, then any gaps are filled with batched `get_vectors()` on the hit keys. **No local `kb/` JSONL reads** are used for retrieval context (an optional appendix cell below can still inspect local files for debugging).

**Legacy global hack.** A naive workaround was: wide top‑k then keep only `source_line ≤ 100`. That does **not** guarantee grammar/style coverage and is **not** used here.

**Source metadata.** `source_file` and `source_line` in vector metadata are the canonical references into the ingested JSONL (see `rag/aws_vectorDB.py`). If a stratum returns zero hits, confirm the exact `source_file` string with the optional **section 4b** probe cell below.

**Credentials:** `VECTORS_AWS_ACCESS_KEY_ID`, `VECTORS_AWS_SECRET_ACCESS_KEY`, and region (see `src/utils/aws_profiles.py`).


## 1. Dependencies (run once per environment)

In [1]:
%pip install sentence-transformers boto3 pandas sacrebleu unbabel-comet --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Project path and imports

In [2]:
import json
import os
import sys
from pathlib import Path

import pandas as pd


def project_root() -> Path:
    """Directory that contains `src/` (works if cwd is repo root or `rag/`)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    raise RuntimeError("Could not find project root (folder with src/retrieval). cd to the repo root or rag/.")


_ROOT = project_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import src.retrieval.s3_vectors_rag as _s3_rag
from src.retrieval.s3_vectors_rag import RetrievedChunk, format_context

print(f"Project root: {_ROOT}")

Project root: c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469


## 3. Configuration

Override with environment variables: `RAG_VECTOR_BUCKET`, `RAG_VECTOR_INDEX`, `RAG_KB_DIR`, `VECTORS_AWS_DEFAULT_REGION`, `EMBED_MODEL`, `GEMINI_TOP_K`, `RAG_MAX_CONTEXT_CHARS`, and per-corpus `RAG_SOURCE_FILE_ENG_JAP`, `RAG_SOURCE_FILE_GEMINI`, `RAG_SOURCE_FILE_GRAMMAR`, `RAG_SOURCE_FILE_STYLE` if metadata strings differ from the defaults below.

In [3]:
# Metadata `source_file` values must match the index exactly (see §4b probe if a stratum returns 0 hits).
# Ingest uses stems like `*_chunks_embedded_full` from `rag/aws_vectorDB.py`; many indexes store `.jsonl` in metadata.
SOURCE_FILE_ENG_JAP = os.environ.get(
    "RAG_SOURCE_FILE_ENG_JAP", "eng_jap_chunks_embedded_full.jsonl"
)
SOURCE_FILE_GEMINI = os.environ.get(
    "RAG_SOURCE_FILE_GEMINI", "gemini_annotated_chunks_embedded_full.jsonl"
)
SOURCE_FILE_GRAMMAR = os.environ.get(
    "RAG_SOURCE_FILE_GRAMMAR", "grammar_chunks_embedded_full.jsonl"
)
SOURCE_FILE_STYLE = os.environ.get(
    "RAG_SOURCE_FILE_STYLE", "style_guide_chunks_embedded_full.jsonl"
)

GEMINI_TOP_K = int(os.environ.get("GEMINI_TOP_K", "2"))  # set to 1 for a single curated hit
MAX_CONTEXT_CHARS = int(os.environ.get("RAG_MAX_CONTEXT_CHARS", "12000"))
EMBED_MODEL = os.environ.get("EMBED_MODEL", "intfloat/multilingual-e5-small")

VECTOR_BUCKET = os.environ.get("RAG_VECTOR_BUCKET", "is469-genai-grp-project")
INDEX_NAME = os.environ.get("RAG_VECTOR_INDEX", "rag-vector-2")
REGION = os.environ.get("VECTORS_AWS_DEFAULT_REGION", "ap-southeast-1")

_kb_env = os.environ.get("RAG_KB_DIR", "").strip()
KB_DIR = Path(_kb_env) if _kb_env else (_ROOT / "kb")

print(f"Bucket: {VECTOR_BUCKET}")
print(f"Index:  {INDEX_NAME}")
print(f"Region: {REGION}")
print(f"kb_dir: {KB_DIR} (exists={KB_DIR.is_dir()})")
print(f"Stratified quotas: eng_jap=3, gemini={GEMINI_TOP_K}, grammar=1, style=1 | embed_model={EMBED_MODEL}")

Bucket: is469-genai-grp-project
Index:  rag-vector-2
Region: ap-southeast-1
kb_dir: c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\kb (exists=True)
Stratified quotas: eng_jap=3, gemini=2, grammar=1, style=1 | embed_model=intfloat/multilingual-e5-small


## 3b. Build S3 Vectors client for chunk text resolution

Uses `s3vectors_client` (same credentials as the retriever) to call `get_vectors()` on the keys
returned by the query. Chunk text is resolved directly from the vector metadata stored in `rag-vector-2` —
no local `kb/` file or separate S3 bucket required.


In [4]:
from src.utils.aws_profiles import s3vectors_client

# Same bucket + index as the retriever
vectors_client = s3vectors_client(region_name=REGION)

def resolve_chunks_from_s3(keys: list[str]) -> dict[str, str]:
    """
    Fetch full vector metadata for a list of keys from rag-vector-2.
    Returns a dict of {key: chunk_text}.
    """
    if not keys:
        return {}
    response = vectors_client.get_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=INDEX_NAME,
        keys=keys,
        returnData=False,       # We only need metadata, not the embedding floats
        returnMetadata=True,
    )
    result = {}
    for vec in response.get("vectors", []):
        key = vec.get("key")
        meta = vec.get("metadata") or {}
        # Try common field names the ingest script may have used
        text = (
            meta.get("text")
            or meta.get("chunk_text")
            or meta.get("content")
        )
        if key and text:
            result[key] = text
    return result

print("✓ s3vectors_client ready — chunk text will be resolved from rag-vector-2 metadata")


✓ s3vectors_client ready — chunk text will be resolved from rag-vector-2 metadata


## 4. KB path mapping and query embedder

The next cell patches `s3_vectors_rag._guess_kb_paths` so metadata names match **this repo’s** `kb/` files, and defines a lazy `SentenceTransformer` encoder (same `query: ` prefix and model as `S3VectorsRAGRetriever` in `s3_vectors_rag.py`).

In [5]:
_embedder = None


def get_embedder():
    """Lazy-load the same E5 query encoder convention as `S3VectorsRAGRetriever`."""
    global _embedder
    if _embedder is None:
        import torch
        from sentence_transformers import SentenceTransformer

        device = os.environ.get("RAG_EMBED_DEVICE", "cpu")
        if device == "cuda" and not torch.cuda.is_available():
            device = "cpu"
        _embedder = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device=device)
        _embedder.max_seq_length = int(
            os.environ.get("MAX_SEQ_LENGTH", "512" if device == "cpu" else "1024")
        )
    return _embedder


def encode_query_vector(query_en: str) -> list[float]:
    import numpy as np

    model = get_embedder()
    text = query_en if query_en.lower().startswith("query:") else f"query: {query_en}"
    vec = model.encode(text, convert_to_numpy=True, show_progress_bar=False)
    return vec.astype("float32").tolist()


print("Query embedder ready (model loads on first `encode_query_vector`).")

Query embedder ready (model loads on first `encode_query_vector`).


## 5. Weighted stratified retrieval

Set `USER_QUERY` to the English sentence (or student question), then run the next cell. It runs **four** filtered `query_vectors` calls (optionally in parallel), merges by quota order, resolves text, and builds `hits_df`.

**Prerequisite:** run **§4** (query embedder) first so `encode_query_vector` exists; otherwise `stratified_retrieve` will raise a clear error.

In [6]:
import time
from concurrent.futures import ThreadPoolExecutor

S3_TEXT_MISSING_PREFIX = "(no chunk text in S3 vector metadata for key="


def _text_from_vector_metadata(meta: dict) -> str:
    """Extract chunk text from query/get_vectors metadata (same field names as `resolve_chunks_from_s3`)."""
    m = meta or {}
    return str(m.get("text") or m.get("chunk_text") or m.get("content") or "")


def _vectors_response_to_chunks(items: list) -> list[RetrievedChunk]:
    """Build chunks from query hits; text comes from query metadata only (get_vectors fills gaps later)."""
    out: list[RetrievedChunk] = []
    for item in items:
        key = str(item.get("key", ""))
        distance = item.get("distance")
        if distance is not None:
            try:
                distance = float(distance)
            except (TypeError, ValueError):
                distance = None
        meta = item.get("metadata") or {}
        source_file = str(meta.get("source_file", ""))
        try:
            source_line = int(meta.get("source_line", -1))
        except (TypeError, ValueError):
            source_line = -1
        text = _text_from_vector_metadata(meta)
        out.append(
            RetrievedChunk(
                key=key,
                distance=distance,
                source_file=source_file,
                source_line=source_line,
                text=text,
            )
        )
    return out


STRATA_SPECS = [
    {"name": "eng_jap", "source_file": SOURCE_FILE_ENG_JAP, "top_k": 3},
    {"name": "gemini_annotated", "source_file": SOURCE_FILE_GEMINI, "top_k": GEMINI_TOP_K},
    {"name": "grammar", "source_file": SOURCE_FILE_GRAMMAR, "top_k": 1},
    {"name": "style_guide", "source_file": SOURCE_FILE_STYLE, "top_k": 1},
]

STRATA_MERGE_ORDER = ["eng_jap", "gemini_annotated", "grammar", "style_guide"]


def stratified_retrieve(query_en: str, *, verbose: bool = True) -> tuple[list[RetrievedChunk], dict[str, str], float]:
    """Run weighted stratified retrieval for one English query; return chunks, stratum map, and wall time in ms."""
    encode = globals().get("encode_query_vector")
    if encode is None:
        raise RuntimeError(
            "encode_query_vector is not defined. Run the §4 cell ('KB path mapping and query embedder') first, "
            "then re-run this cell."
        )
    t0 = time.perf_counter()
    qvec = encode(query_en)

    def _query_one_stratum(spec: dict) -> tuple[str, list[RetrievedChunk]]:
        name = spec["name"]
        sf = spec["source_file"]
        k = int(spec["top_k"])
        flt = {"source_file": {"$eq": sf}}
        try:
            resp = vectors_client.query_vectors(
                vectorBucketName=VECTOR_BUCKET,
                indexName=INDEX_NAME,
                topK=k,
                queryVector={"float32": qvec},
                returnMetadata=True,
                returnDistance=True,
                filter=flt,
            )
        except Exception as e:
            if verbose:
                print(f"[{name}] query_vectors failed ({sf!r}): {e}")
            return name, []
        vecs = resp.get("vectors") or []
        if not vecs and verbose:
            print(f"[{name}] warning: 0 hits for source_file={sf!r} — check SOURCE_FILE_* / unfiltered query sample.")
        chs = _vectors_response_to_chunks(vecs)
        if verbose:
            print(f"[{name}] retrieved {len(chs)} chunk(s) (requested top_k={k}).")
        return name, chs

    with ThreadPoolExecutor(max_workers=4) as ex:
        stratum_results = list(ex.map(_query_one_stratum, STRATA_SPECS))

    by_name = dict(stratum_results)
    chunks: list[RetrievedChunk] = []
    chunk_stratum: dict[str, str] = {}
    seen_keys: set[str] = set()
    for nm in STRATA_MERGE_ORDER:
        for c in by_name.get(nm, []):
            if c.key and c.key not in seen_keys:
                seen_keys.add(c.key)
                chunks.append(c)
                chunk_stratum[c.key] = nm

    keys_missing_text = [c.key for c in chunks if c.key and not (c.text or "").strip()]
    if keys_missing_text and verbose:
        print(f"\nFetching text for {len(keys_missing_text)} chunk(s) via get_vectors...")
    s3_text_map = resolve_chunks_from_s3(keys_missing_text) if keys_missing_text else {}
    for chunk in chunks:
        if chunk.key and not (chunk.text or "").strip():
            chunk.text = s3_text_map.get(chunk.key, "")

    for chunk in chunks:
        if chunk.key and not (chunk.text or "").strip():
            chunk.text = (
                f"{S3_TEXT_MISSING_PREFIX}{chunk.key}; "
                f"source_file={chunk.source_file} line={chunk.source_line})"
            )

    retrieval_ms = round((time.perf_counter() - t0) * 1000, 1)
    return chunks, chunk_stratum, retrieval_ms


print("stratified_retrieve() defined (used by §5 demo and §6 eval batch).")

stratified_retrieve() defined (used by §5 demo and §6 eval batch).


In [7]:
import pandas as pd

USER_QUERY = "Example: How do I express polite requests in Japanese?"

chunks, chunk_stratum, retrieval_ms = stratified_retrieve(USER_QUERY, verbose=True)
print(f"\nretrieval_ms (stratified query + merge + get_vectors): {retrieval_ms}")

context = format_context(chunks, max_chars=MAX_CONTEXT_CHARS)

print("\n=== Formatted context (for prompting) ===\n")
print(context)

# ── DataFrame summary ────────────────────────────────────────────────────────
_preview_chars = 320
_rows: list[dict] = []
for i, c in enumerate(chunks, start=1):
    t = c.text or ""
    unresolved = t.startswith("(no local text resolved") or t.startswith(S3_TEXT_MISSING_PREFIX)
    preview = t if len(t) <= _preview_chars else f"{t[: _preview_chars - 1]}…"
    _rows.append(
        {
            "rank": i,
            "stratum": chunk_stratum.get(c.key, ""),
            "distance": c.distance,
            "key": c.key,
            "source_file": c.source_file,
            "source_line": c.source_line,
            "resolved": not unresolved,
            "text_preview": preview,
            "chunk_text": t,
        }
    )

hits_df = pd.DataFrame(_rows)
if hits_df["distance"].notna().any():
    hits_df["distance"] = pd.to_numeric(hits_df["distance"], errors="coerce").round(6)

hits_summary = hits_df[
    ["rank", "stratum", "distance", "key", "source_file", "source_line", "resolved", "text_preview"]
]
print("\n=== Retrieval hits (summary DataFrame: `hits_df`, display below) ===\n")
hits_summary


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

[style_guide] warning: 0 hits for source_file='style_guide_chunks_embedded_full.jsonl' — check SOURCE_FILE_* / unfiltered query sample.
[style_guide] retrieved 0 chunk(s) (requested top_k=1).
[eng_jap] retrieved 3 chunk(s) (requested top_k=3).
[grammar] retrieved 1 chunk(s) (requested top_k=1).
[gemini_annotated] retrieved 2 chunk(s) (requested top_k=2).

Fetching text for 6 chunk(s) via get_vectors...

retrieval_ms (stratified query + merge + get_vectors): 25771.8

=== Formatted context (for prompting) ===

[eng_jap_chunks_embedded_full.jsonl L8432]
(no chunk text in S3 vector metadata for key=eng_jap_chunks_embedded_full:8431; source_file=eng_jap_chunks_embedded_full.jsonl line=8432)

---

[eng_jap_chunks_embedded_full.jsonl L79]
(no chunk text in S3 vector metadata for key=eng_jap_chunks_embedded_full:78; source_file=eng_jap_chunks_embedded_full.jsonl line=79)

---

[eng_jap_chunks_embedded_full.jsonl L30252]
(no chunk text in S3 vector metadata for key=eng_jap_chunks_embedded_full:

,rank,stratum,distance,key,source_file,source_line,resolved,text_preview
0,1,eng_jap,0.103171,eng_jap_chunks_embedded_full:8431,eng_jap_chunks_embedded_full.jsonl,8432,False,(no chunk text in S3 vector metadata for key=e...
1,2,eng_jap,0.104292,eng_jap_chunks_embedded_full:78,eng_jap_chunks_embedded_full.jsonl,79,False,(no chunk text in S3 vector metadata for key=e...
2,3,eng_jap,0.105754,eng_jap_chunks_embedded_full:30251,eng_jap_chunks_embedded_full.jsonl,30252,False,(no chunk text in S3 vector metadata for key=e...
3,4,gemini_annotated,0.120494,gemini_annotated_chunks_embedded_full:410,gemini_annotated_chunks_embedded_full.jsonl,411,False,(no chunk text in S3 vector metadata for key=g...
4,5,gemini_annotated,0.123998,gemini_annotated_chunks_embedded_full:101,gemini_annotated_chunks_embedded_full.jsonl,102,False,(no chunk text in S3 vector metadata for key=g...
5,6,grammar,0.163404,grammar_chunks_embedded_full:29,grammar_chunks_embedded_full.jsonl,30,False,(no chunk text in S3 vector metadata for key=g...


In [8]:
# Source-line sanity check (Eng↔Jap skew + why local resolution disagrees with labels)
import json

chunks_file = _ROOT / "kb" / "eng_jap_chunks.jsonl"
print(f"{chunks_file.name} exists: {chunks_file.exists()}")

with open(chunks_file, encoding="utf-8") as f:
    lines = f.readlines()

print(
    f"Total JSONL lines: {len(lines):,} — global top-k is dominated by this file; "
    "this notebook uses per-corpus filtered retrieval instead of a line band.\n"
)

print("First rows in the early band (typical context after filtering):")
for lineno in range(1, min(4, len(lines) + 1)):
    record = json.loads(lines[lineno - 1])
    snippet = json.dumps(record, ensure_ascii=False)[:180]
    print(f"  L{lineno}: {snippet}…")

# Deep rows exist on disk; "unresolved" in the retriever is usually path/name mismatch
# (metadata says eng_jap_chunks_embedded_full.jsonl but repo line lives in eng_jap_chunks.jsonl).
for lineno in [8432, 30252]:
    if lineno <= len(lines):
        record = json.loads(lines[lineno - 1])
        snippet = json.dumps(record, ensure_ascii=False)[:160]
        print(f"\nDeep row still valid at L{lineno} in eng_jap_chunks.jsonl: {snippet}…")

eng_jap_chunks.jsonl exists: True
Total JSONL lines: 46,616 — global top-k is dominated by this file; this notebook uses per-corpus filtered retrieval instead of a line band.

First rows in the early band (typical context after filtering):
  L1: {"chunk_id": "engjap_chunk_000000_000007", "start_row": 0, "end_row": 7, "n_pairs": 8, "chunk_text": "[1] EN: Let's try something.\n[1] JA: 何かしてみましょう。\n\n[2] EN: I have to go to sl…
  L2: {"chunk_id": "engjap_chunk_000006_000013", "start_row": 6, "end_row": 13, "n_pairs": 8, "chunk_text": "[1] EN: The password is \"Muiriel\".\n[1] JA: パスワードは「Muiriel」です。\n\n[2] EN: I…
  L3: {"chunk_id": "engjap_chunk_000012_000019", "start_row": 12, "end_row": 19, "n_pairs": 8, "chunk_text": "[1] EN: I will be back soon.\n[1] JA: すぐ戻る。\n\n[2] EN: I'm at a loss for wor…

Deep row still valid at L8432 in eng_jap_chunks.jsonl: {"chunk_id": "engjap_chunk_050586_050593", "start_row": 50586, "end_row": 50593, "n_pairs": 8, "chunk_text": "[1] EN: I'm very pleased to me

## 6. Evaluation / metrics

This section batches **retrieval** over `kb/eng-jap.tsv` (same column layout as `evaluate_outputs._load_tsv_eval_set`: id, `source_en`, column 3 ignored, `reference_ja`), writes [results/query_pipeline_eval.jsonl](results/query_pipeline_eval.jsonl), and calls `evaluate_rows` from [rag/advanced_rag/evaluate_outputs.py](rag/advanced_rag/evaluate_outputs.py).

**What you get here**

- **Retrieval:** `retrieval_hit_at_k`, `retrieval_recall_at_k`, `retrieval_eval_samples` (chunks are rebuilt into `retrieval_eval` using `KB_DIR` assets).
- **Terminology / error-ID:** Only when KB annotations and row fields align; `prediction_ja` is left empty in this notebook, so **BLEU, chrF++, and COMET are skipped** (no hypothesis text). For a full S3-style report (latency vs generation, `coverage_score`, `total_rewrite_steps`, COMET, etc.), run the agentic Modal pipeline or add generation in a separate notebook.

Set `MAX_EVAL_SAMPLES` below (and optionally `EVAL_TSV`) before running.

In [9]:
MAX_EVAL_SAMPLES = int(os.environ.get("QP_EVAL_MAX_SAMPLES", "50"))
EVAL_TSV = Path(os.environ.get("QP_EVAL_TSV", str(KB_DIR / "eng-jap.tsv")))
EVAL_JSONL = _ROOT / "results" / "query_pipeline_eval.jsonl"


def load_eval_tsv(path: Path, max_samples: int) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []
    with path.open("r", encoding="utf-8-sig") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 4:
                continue
            item_id = parts[0].strip()
            source_en = parts[1].strip()
            reference_ja = parts[3].strip()
            if not source_en or not reference_ja:
                continue
            rows.append(
                {
                    "id": f"engjap-{item_id}",
                    "source_en": source_en,
                    "reference_ja": reference_ja,
                }
            )
            if len(rows) >= max_samples:
                break
    return rows


def chunk_to_eval_dict(chunk: RetrievedChunk) -> dict:
    text = str(chunk.text or "")
    return {
        "text": text,
        "text_preview": (text[:240] + "...") if len(text) > 240 else text,
        "stratum": None,
        "rerank_score": None,
        "distance": chunk.distance,
        "key": chunk.key,
        "source_file": chunk.source_file,
        "source_line": chunk.source_line,
    }


def build_eval_jsonl(*, verbose_every: int = 10) -> Path:
    if not EVAL_TSV.is_file():
        raise FileNotFoundError(f"Eval TSV not found: {EVAL_TSV}")

    eval_rows = load_eval_tsv(EVAL_TSV, MAX_EVAL_SAMPLES)
    if not eval_rows:
        raise RuntimeError(f"No valid rows in {EVAL_TSV}")

    EVAL_JSONL.parent.mkdir(parents=True, exist_ok=True)
    written: list[dict] = []

    for i, sample in enumerate(eval_rows, start=1):
        if verbose_every and i % verbose_every == 1:
            print(f"[eval batch] {i}/{len(eval_rows)} id={sample['id']!r}")

        chunks_i, _stratum_map, r_ms = stratified_retrieve(sample["source_en"], verbose=False)
        rec = {
            "id": sample["id"],
            "source_en": sample["source_en"],
            "reference_ja": sample["reference_ja"],
            "prediction_ja": "",
            "variant": "query_pipeline",
            "retrieval_chunks": [chunk_to_eval_dict(c) for c in chunks_i],
            "latency_ms": r_ms,
            "retrieval_ms": r_ms,
        }
        written.append(rec)

    with EVAL_JSONL.open("w", encoding="utf-8") as f:
        for rec in written:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"Wrote {len(written)} row(s) -> {EVAL_JSONL}")
    return EVAL_JSONL


build_eval_jsonl()

[eval batch] 1/50 id='engjap-1276'
[eval batch] 11/50 id='engjap-1284'
[eval batch] 21/50 id='engjap-1291'
[eval batch] 31/50 id='engjap-1302'
[eval batch] 41/50 id='engjap-1311'
Wrote 50 row(s) -> c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\results\query_pipeline_eval.jsonl


WindowsPath('c:/Users/Jonathan/Desktop/Y4S2/GenAI/is469/results/query_pipeline_eval.jsonl')

In [10]:
import importlib.util

_spec = importlib.util.spec_from_file_location(
    "qp_evaluate_outputs",
    _ROOT / "rag" / "advanced_rag" / "evaluate_outputs.py",
)
_mod = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_mod)
evaluate_rows = _mod.evaluate_rows

_rows_for_metrics = []
with EVAL_JSONL.open("r", encoding="utf-8-sig") as f:
    for line in f:
        line = line.strip()
        if line:
            _rows_for_metrics.append(json.loads(line))

metrics = evaluate_rows(_rows_for_metrics, kb_dir=KB_DIR, output_path=EVAL_JSONL)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\.venv\Lib\site-packages\torchmetrics\utilities\imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

LICENSE: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Encoder model frozen.
c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\.venv\Lib\site-packages\pytorch_lightning\core\saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


{
  "output_path": "c:\\Users\\Jonathan\\Desktop\\Y4S2\\GenAI\\is469\\results\\query_pipeline_eval.jsonl",
  "num_samples": 50,
  "avg_latency_ms": 318.68,
  "avg_retrieval_ms": 318.68,
  "avg_coverage_score": null,
  "bleu": 0.0,
  "bleu_tokenizer": "char",
  "chrfpp": 0.0,
  "comet": 0.4543,
  "retrieval_eval_samples": 50,
  "retrieval_hit_at_k": 0.0,
  "retrieval_recall_at_k": 0.0,
  "terminology_eval_samples": 0,
  "terminology_accuracy": null,
  "error_id_eval_samples": 0,
  "error_binary_f1": null,
  "error_category_macro_f1": null
}
